# ModelForge Lite — Phase 2: Baseline (Zero-Shot) Evaluation

This notebook loads the base model with no fine-tuning and no retrieval, and runs it on the held-out `eval.csv` questions. This is Variant 1 of 4 — the baseline everything else gets compared against.

Run in Colab with the T4 GPU enabled (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!pip install -q transformers accelerate datasets huggingface_hub pandas

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Pull the eval split from Phase 1

Replace `YOUR_HF_USERNAME` with your actual Hugging Face username (same one used in Phase 1).

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

HF_USERNAME = "YOUR_HF_USERNAME"
dataset_repo = f"{HF_USERNAME}/modelforge-lite-support-data"

eval_path = hf_hub_download(repo_id=dataset_repo, filename="eval.csv", repo_type="dataset")
eval_df = pd.read_csv(eval_path)
print(f"Loaded {len(eval_df)} eval questions")
eval_df.head()

## 2. Load the base model

We're using `Qwen/Qwen2.5-0.5B-Instruct` — small enough to run comfortably on a free Colab GPU, instruction-tuned so it can follow a support-agent style prompt out of the box.

If you'd rather use TinyLlama instead, swap the model name below to `TinyLlama/TinyLlama-1.1B-Chat-v1.0` — everything else in this notebook works the same either way.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("Model loaded.")

## 3. Write the generation function

We use a simple system prompt so the model behaves like a support agent. We are NOT giving it any company-specific knowledge here — that's the point of this baseline. It only knows what it learned during its original pretraining.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful customer support assistant. "
    "Answer the customer's question clearly and concisely."
)

def generate_answer(question, max_new_tokens=150):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    import time
    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    latency = time.time() - start

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.strip(), latency

# quick sanity check
test_answer, test_latency = generate_answer("How do I track my order?")
print(f"Answer: {test_answer}\n\nLatency: {test_latency:.2f}s")

## 4. Run the base model over all eval questions

This is the actual baseline run. Each answer plus its latency gets saved — latency matters later because it's one of the three things we compare across all four variants.

In [ ]:
results = []
for i, row in eval_df.iterrows():
    question = row["instruction"]  # adjust column name if different in your dataset
    answer, latency = generate_answer(question)
    results.append({
        "question": question,
        "intent": row.get("intent", ""),
        "base_model_answer": answer,
        "base_model_latency_sec": round(latency, 3),
    })
    print(f"[{i+1}/{len(eval_df)}] done")

baseline_df = pd.DataFrame(results)
baseline_df.to_csv("baseline_results.csv", index=False)
baseline_df.head()

## 5. Push results to your Hugging Face dataset repo

This keeps every variant's results in one place, so the evaluation harness in Phase 7 can pull them all together and score them side by side.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="baseline_results.csv",
    path_in_repo="results/baseline_results.csv",
    repo_id=dataset_repo,
    repo_type="dataset",
)
print("Uploaded baseline results.")

## Done — Phase 2 checklist

- [ ] Base model loaded (no fine-tuning, no retrieval)
- [ ] Ran on all held-out eval questions
- [ ] Saved answers + latency for each question
- [ ] Pushed `baseline_results.csv` to your Hugging Face dataset repo

## What to look at before moving on
Skim a few of the answers in `baseline_results.csv`. You're looking for generic, plausible-sounding answers that lack any company-specific detail (exact refund windows, specific policy names, etc.) — that's expected, and it's exactly the gap RAG and fine-tuning are meant to close. Note down 2-3 examples where the answer is vague or slightly off — these make great "before" examples to contrast with later variants on the call.

Next: Phase 3 — Base model + RAG.